# Amazon Baby — Qwen3-0.6B QLoRA Fine-tune

**Dataset:** `roopalik/amazon-baby-dataset`

**Input:** Product context + Review → **Output:** Rating 1–5

**Final artifact:** `/kaggle/working/amazon-baby-qwen3-finetuned.zip` (merged standalone model)

Settings → Accelerator → **GPU** required.

In [ ]:
# torch'u upgrade etme
!pip install -q "transformers>=4.52.0" "peft>=0.14.0" datasets accelerate safetensors "torchao>=0.16.0" bitsandbytes

In [ ]:
import os, gc, glob, shutil
from pathlib import Path
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
import torch
import pandas as pd
from torch.utils.data import Sampler, DataLoader
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, Trainer, TrainingArguments
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "Qwen/Qwen3-0.6B"
ADAPTER_DIR = "/kaggle/working/amazon-baby-lora"
FINAL_MODEL_DIR = "/kaggle/working/amazon-baby-qwen3-finetuned"
FINAL_ZIP = "/kaggle/working/amazon-baby-qwen3-finetuned.zip"
EPOCHS, LEARNING_RATE, GRAD_ACCUMULATION, MODEL_CONTEXT = 1, 1e-4, 4, 32768
MAX_BATCH_TOKENS, MAX_BATCH_SIZE = 6144, 4
TOKENIZE_BATCH, TOKENIZE_WORKERS = 256, 4

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

if not torch.cuda.is_available():
    raise RuntimeError("GPU bulunamadi. Kaggle -> Settings -> Accelerator -> GPU sec.")
print("GPU:", torch.cuda.get_device_name(0))
bf16_supported = torch.cuda.is_bf16_supported()
compute_dtype = torch.bfloat16 if bf16_supported else torch.float16
print("Compute dtype:", compute_dtype)

csv_files = glob.glob("/kaggle/input/**/*.csv", recursive=True)
print("\nCSV files:")
for file in csv_files: print(file)

DATA_PATH, SCHEMA = None, None
for file in csv_files:
    try:
        test = pd.read_csv(file, nrows=10)
        cols = {str(c).strip().lower(): c for c in test.columns}
        colset = set(cols.keys())
        if {"name", "review", "rating"}.issubset(colset):
            DATA_PATH, SCHEMA = file, "legacy"; break
        if {"reviewtext", "overall"}.issubset(colset):
            DATA_PATH, SCHEMA = file, "amazon_baby_real"; break
    except Exception: pass

if DATA_PATH is None:
    raise FileNotFoundError("CSV bulunamadi: (name,review,rating) veya (reviewText,overall)")
print("\nDataset:", DATA_PATH, "| Schema:", SCHEMA)

df = pd.read_csv(DATA_PATH)
raw_cols = {str(c).strip().lower(): c for c in df.columns}
if SCHEMA == "legacy":
    df = df.rename(columns={raw_cols["name"]: "name", raw_cols["review"]: "review", raw_cols["rating"]: "rating"})
else:
    name_src = raw_cols.get("summary") or raw_cols.get("title") or raw_cols.get("asin")
    df = df.rename(columns={name_src: "name", raw_cols["reviewtext"]: "review", raw_cols["overall"]: "rating"})

print("Original:", df.shape)
df["rating"] = pd.to_numeric(df["rating"], errors="coerce")
df = df[df["rating"].isin([1,2,3,4,5])].copy()
df["rating"] = df["rating"].astype(int)
df["name"] = df["name"].fillna("").astype(str)
df["review"] = df["review"].fillna("").astype(str)
df.reset_index(drop=True, inplace=True)
print("Training samples:", len(df))
print(df["rating"].value_counts().sort_index())

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

INSTRUCTION = """Predict the original rating assigned to this Amazon baby product review.
The rating must be exactly one integer from: 1, 2, 3, 4, 5
Carefully read the complete product name and review.
Return only the rating number."""

def create_prompt(row):
    return f"""### Task\n{INSTRUCTION}\n\n### Product\n{row['name']}\n\n### Review\n{row['review']}\n\n### Rating\n"""

df["prompt"] = df.apply(create_prompt, axis=1)
df["completion"] = df["rating"].astype(str)
print("\nEXAMPLE:\n", df.iloc[0]["prompt"], "EXPECTED:", df.iloc[0]["completion"])

print("\nChecking token lengths...")
max_tokens, max_index = 0, 0
for start in range(0, len(df), 512):
    end = min(start + 512, len(df))
    texts = [p + c for p, c in zip(df["prompt"].iloc[start:end], df["completion"].iloc[start:end])]
    encoded = tokenizer(texts, truncation=False, padding=False, add_special_tokens=True, return_length=True)
    lengths = encoded["length"]
    current_max = max(lengths)
    if current_max > max_tokens:
        max_tokens = current_max
        max_index = start + lengths.index(current_max)
    if start % 10000 == 0: print(f"{start:,} / {len(df):,}")
print("Longest example:", max_tokens, "tokens")
if max_tokens > MODEL_CONTEXT:
    raise RuntimeError(f"En uzun review {max_tokens} token > {MODEL_CONTEXT}. Row: {max_index}")

train_dataset = Dataset.from_dict({
    "text": (df["prompt"] + df["completion"]).tolist(),
})

print("\nLoading:", MODEL_ID, "(4-bit QLoRA + T4)")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=compute_dtype,
    attn_implementation="sdpa",
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM", target_modules="all-linear")
model = get_peft_model(model, lora_config)

def tokenize_batch(examples):
    out = tokenizer(examples["text"], truncation=False, add_special_tokens=True)
    out["length"] = [len(x) for x in out["input_ids"]]
    return out

train_dataset = train_dataset.map(
    tokenize_batch,
    batched=True,
    batch_size=TOKENIZE_BATCH,
    num_proc=TOKENIZE_WORKERS,
    remove_columns=["text"],
    desc="Tokenizing",
)
train_dataset = train_dataset.sort("length")
print("Dataset sorted by length for efficient batching")

RESPONSE_TEMPLATE = "### Rating\n"
TEMPLATE_IDS = tokenizer.encode(RESPONSE_TEMPLATE, add_special_tokens=False)
TEMPLATE_LEN = len(TEMPLATE_IDS)

def completion_only_collator(features):
    max_len = max(len(f["input_ids"]) for f in features)
    input_ids, attention_mask, labels = [], [], []
    for f in features:
        ids = f["input_ids"]
        pad_len = max_len - len(ids)
        input_ids.append(ids + [tokenizer.pad_token_id] * pad_len)
        attention_mask.append([1] * len(ids) + [0] * pad_len)
        lab = ids[:]
        masked = False
        for idx in range(len(lab) - TEMPLATE_LEN + 1):
            if lab[idx : idx + TEMPLATE_LEN] == TEMPLATE_IDS:
                lab = [-100] * (idx + TEMPLATE_LEN) + lab[idx + TEMPLATE_LEN :]
                masked = True
                break
        if not masked:
            lab = [-100] * len(lab)
        labels.append(lab + [-100] * pad_len)
    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
    }

class TokenBudgetBatchSampler(Sampler):
    def __init__(self, lengths, max_batch_tokens, max_batch_size):
        self.batches = []
        batch, batch_max = [], 0
        for idx, ln in enumerate(lengths):
            new_max = max(batch_max, ln)
            if batch and ((len(batch) + 1) * new_max > max_batch_tokens or len(batch) >= max_batch_size):
                self.batches.append(batch)
                batch, batch_max = [idx], ln
            else:
                batch.append(idx)
                batch_max = new_max
        if batch:
            self.batches.append(batch)

    def __iter__(self):
        return iter(self.batches)

    def __len__(self):
        return len(self.batches)

lengths = train_dataset["length"]
batch_sampler = TokenBudgetBatchSampler(lengths, MAX_BATCH_TOKENS, MAX_BATCH_SIZE)
print(f"Dynamic batches: {len(batch_sampler)} (max {MAX_BATCH_SIZE} x {MAX_BATCH_TOKENS} tokens)")

training_args = TrainingArguments(
    output_dir=ADAPTER_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.01,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    bf16=bf16_supported,
    fp16=not bf16_supported,
    logging_steps=50,
    report_to="none",
    save_strategy="steps",
    save_steps=400,
    save_total_limit=2,
    dataloader_num_workers=0,
    seed=42,
    remove_unused_columns=False,
)

class VariableBatchTrainer(Trainer):
    def get_train_dataloader(self):
        return DataLoader(
            self.train_dataset,
            batch_sampler=batch_sampler,
            collate_fn=self.data_collator,
            num_workers=0,
            pin_memory=True,
        )

trainer = VariableBatchTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=completion_only_collator,
    processing_class=tokenizer,
)
trainer.model.print_trainable_parameters()
print("\n" + "="*60 + "\nFINE-TUNING STARTED\n" + "="*60)
checkpoints = sorted(Path(ADAPTER_DIR).glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]))
resume = str(checkpoints[-1]) if checkpoints else None
if resume:
    print("Resuming from:", resume)
trainer.train(resume_from_checkpoint=resume)
print("\n" + "="*60 + "\nFINE-TUNING FINISHED\n" + "="*60)

trainer.save_model(ADAPTER_DIR); tokenizer.save_pretrained(ADAPTER_DIR)
del trainer, model; gc.collect(); torch.cuda.empty_cache()

print("\nReloading base model for merge...")
base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=compute_dtype, device_map="cpu", low_cpu_mem_usage=True)
merged_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR).merge_and_unload()

os.makedirs(FINAL_MODEL_DIR, exist_ok=True)
merged_model.save_pretrained(FINAL_MODEL_DIR, safe_serialization=True)
tokenizer.save_pretrained(FINAL_MODEL_DIR)
if os.path.exists(FINAL_ZIP): os.remove(FINAL_ZIP)
shutil.make_archive(FINAL_ZIP.replace(".zip", ""), "zip", FINAL_MODEL_DIR)
print("\nDOWNLOAD:", FINAL_ZIP, "|", round(os.path.getsize(FINAL_ZIP)/1024/1024, 2), "MB")
print("FINISHED - standalone model, LoRA adapter gerekmez.")